In [1]:
import os

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

import sys

sclembas = '/home/hmbaghda/Projects/scLEMBAS'
sys.path.insert(1, os.path.join(sclembas))
from scLEMBAS import parse_network, io

/nobackup/users/hmbaghda/Software/miniforge3/envs/scLEMBAS/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
n_cores = 12
os.environ["OMP_NUM_THREADS"] = str(n_cores)
os.environ["MKL_NUM_THREADS"] = str(n_cores)
os.environ["OPENBLAS_NUM_THREADS"] = str(n_cores)
os.environ["VECLIB_MAXIMUM_THREADS"] = str(n_cores)
os.environ["NUMEXPR_NUM_THREADS"] = str(n_cores)

seed = 888
data_path = '/nobackup/users/hmbaghda/scLEMBAS/analysis'

In [3]:
tf_adata = io.read_tfad(file_name = os.path.join(data_path, 'interim', 'Kang_tf_activity.h5ad'))

Load and parse input signaling network:

In [3]:
source_label = 'source_genesymbol'
target_label = 'target_genesymbol'
weight_label = 'mode_of_action'
stimulation_label = 'consensus_stimulation'
inhibition_label = 'consensus_inhibition'

In [5]:
sn_ppis = parse_network.load_network('omnipath', organism = 'human', static = True)
sn_ppis = parse_network.correct_network(sn_ppis = sn_ppis,
                                        source_label = source_label, target_label = target_label,
                                        stimulation_label = stimulation_label, inhibition_label = inhibition_label)

/home/hmbaghda/Projects/scLEMBAS/scLEMBAS/parse_network.py:162: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  unique_vals = pd.concat([unique_vals, dup_int], axis = 0)


In [6]:
ligand_labels = ['IFNB1']
tf_labels = tf_adata.var.index.unique().tolist()

Retain a signaling network of nodes with full paths from IFNB1 to the inferred TFs, first filtering the network for interactions with atleast one reference and a curation effort of 2: 

In [7]:
sn_ppis = parse_network.extract_network(sn_ppis = sn_ppis.copy(), 
                                        curation_effort_thresh = 2, 
                          n_references_thresh = 1,
                          resources = 'all', 
                          drop_self = True, 
                          source_label = source_label, 
                          target_label = target_label,
                          verbose = False)
sn_ppis, ligand_connections = parse_network.create_connected_network(sn_ppis = sn_ppis, 
                                                       ligand_labels = ligand_labels, 
                                                       tf_labels = tf_labels, 
                                                       source_label = source_label, 
                                                       target_label = target_label,
                                                       path_finder = 'connected')

all_nodes_ = sorted(set(sn_ppis[source_label].tolist() + sn_ppis[target_label].tolist()))
print('The pruned network has {} interactions between {} nodes'.format(sn_ppis.shape[0], len(all_nodes_)))  
print('{} of {} TFs are retained in the final signaling network'.format(len(ligand_connections['IFNB1']), 
                                                                        len(tf_labels)))

The pruned network has 19248 interactions between 3440 nodes
216 of 423 TFs are retained in the final signaling network


Format network as needed for model building:

In [8]:
sn_ppis = parse_network.format_network(sn_ppis, weight_label, stimulation_label, inhibition_label) 

In [9]:
sn_ppis[[source_label, target_label, weight_label, stimulation_label, inhibition_label]].head()

,source_genesymbol,target_genesymbol,mode_of_action,consensus_stimulation,consensus_inhibition
0,CALM1,TRPC1,-1.0,False,True
1,CALM3,TRPC1,-1.0,False,True
2,CALM2,TRPC1,-1.0,False,True
3,CAV1,TRPC1,1.0,True,False
5,MDFI,TRPC1,-1.0,False,True


In [10]:
sn_ppis.to_csv(os.path.join(data_path, 'processed', 'Kang_sn_ppis.csv'))